# Decision Tree Classifier

In [99]:
import os
import soundfile
import numpy as np

genre = ['Rock', 'Singer-Songwriter', 'Bossa Nova', 'Jazz', 'Funk']
genre_code = {'Rock': 'Rock', 'SS': 'Singer-Songwriter', 'BN': 'Bossa Nova', 'Jazz': 'Jazz', 'Funk': 'Funk'}

In [100]:
import librosa

def get_mel_spectrogram(y, sr):
    return librosa.feature.melspectrogram(y=y, sr=sr)

# Converting .wav files to .npy files
count_saved = 0
cut_seconds = 5

X = []
Y = []

def get_label(filepath):
    for label in genre_code:
        if label in filepath:
            return genre_code[label]
    return ""

for file in os.listdir('data'):
    data, samplerate = soundfile.read(os.path.join('data', file))
    data = np.mean(data, axis=1)
    no_clips = int(len(data)/(cut_seconds*samplerate))

    for i in range (no_clips):
        cut_data = data[i:i+cut_seconds*samplerate]

        label = get_label(file)
        if label == " ":
            print("ERROR")
            break
        mel = get_mel_spectrogram(cut_data, samplerate)
        X.append(mel)
        Y.append(label)

    print(f"file {file} has {no_clips} clips and is ended")

print(Y.count(" "))

print(count_saved)


file 00_BN1-129-Eb_comp_hex.wav has 4 clips and is ended
file 00_BN1-129-Eb_solo_hex.wav has 4 clips and is ended
file 00_BN1-147-Gb_comp_hex.wav has 3 clips and is ended
file 00_BN1-147-Gb_solo_hex.wav has 3 clips and is ended
file 00_BN2-131-B_comp_hex.wav has 5 clips and is ended
file 00_BN2-131-B_solo_hex.wav has 5 clips and is ended
file 00_BN2-166-Ab_comp_hex.wav has 4 clips and is ended
file 00_BN2-166-Ab_solo_hex.wav has 4 clips and is ended
file 00_BN3-119-G_comp_hex.wav has 6 clips and is ended
file 00_BN3-119-G_solo_hex.wav has 6 clips and is ended
file 00_BN3-154-E_comp_hex.wav has 4 clips and is ended
file 00_BN3-154-E_solo_hex.wav has 4 clips and is ended
file 00_Funk1-114-Ab_comp_hex.wav has 5 clips and is ended
file 00_Funk1-114-Ab_solo_hex.wav has 5 clips and is ended
file 00_Funk1-97-C_comp_hex.wav has 5 clips and is ended
file 00_Funk1-97-C_solo_hex.wav has 5 clips and is ended
file 00_Funk2-108-Eb_comp_hex.wav has 7 clips and is ended
file 00_Funk2-108-Eb_solo_hex.w

# CNN class

In [101]:
from torch import nn, Tensor
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(CNN, self).__init__()

        # Blok 1
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Blok 2
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Blok 3
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(in_features=32 * 16 * 53, out_features=32)
        self.drop1 = nn.Dropout(p=0.5)

        self.fc2 = nn.Linear(in_features=32, out_features=32)
        self.drop2 = nn.Dropout(p=0.5)

        self.out = nn.Linear(in_features=32, out_features=out_channels)

    def forward(self, x):
        # Blok 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Blok 2
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))

        # Blok 3
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        # Warstwy klasyfikujące
        x = self.flatten(x)

        x = F.relu(self.fc1(x))
        x = self.drop1(x)

        x = F.relu(self.fc2(x))
        x = self.drop2(x)

        x = self.out(x)

        return x

In [102]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

print(np.shape(X))
print(np.shape(Y))

X = np.array(X)
# Y = np.array(Y)

label_encoder = LabelEncoder()
Y_enc = label_encoder.fit_transform(np.array(Y))
print(label_encoder.classes_)

import torch
import torch.nn as nn
import torch.optim as optim

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    """
    Główna funkcja do trenowania i walidacji modelu.
    """

    # Przenosimy model na odpowiednie urządzenie (CPU lub GPU)
    model.to(device)

    print(f"Rozpoczynanie treningu na {device}...")

    for epoch in range(num_epochs):

        # --- Faza Treningu ---
        model.train() # Ustawia model w tryb treningu (włącza dropout, batchnorm itp.)
        running_train_loss = 0.0

        for inputs, labels in train_loader:
            # Przenosimy dane na urządzenie
            inputs, labels = inputs.to(device), labels.to(device)

            # 1. Wyzerowanie gradientów
            optimizer.zero_grad()

            # 2. Forward pass (przepuszczenie danych przez model)
            outputs = model(inputs)

            # 3. Obliczenie straty
            loss = criterion(outputs, labels)

            # 4. Backward pass (obliczenie gradientów)
            loss.backward()

            # 5. Aktualizacja wag
            optimizer.step()

            # Zbieramy statystyki
            running_train_loss += loss.item() * inputs.size(0)

        avg_train_loss = running_train_loss / len(train_loader.dataset)

        # --- Faza Walidacji ---
        model.eval() # Ustawia model w tryb ewaluacji (wyłącza dropout, batchnorm itp.)
        running_val_loss = 0.0
        correct_preds = 0
        total_preds = 0

        # Wyłączamy obliczanie gradientów, aby oszczędzić pamięć i przyspieszyć obliczenia
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item() * inputs.size(0)

                # Obliczanie dokładności
                # torch.max zwraca (wartość, indeks) wzdłuż danego wymiaru
                _, predicted = torch.max(outputs.data, 1)
                total_preds += labels.size(0)
                correct_preds += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader.dataset)
        val_accuracy = (correct_preds / total_preds) * 100

        # Wyświetlanie wyników po każdej epoce
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Accuracy: {val_accuracy:.2f}%")

    print("Trening zakończony.")
    return model # Zwracamy wytrenowany model

# Załóżmy, że masz już:
# train_loader - DataLoader dla danych treningowych
# val_loader - DataLoader dla danych walidacyjnych
# CNN - klasa twojego modelu (ta, którą podałeś wcześniej)

# 1. Ustawienie urządzenia (GPU, jeśli dostępne)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train, X_val, y_train, y_val = train_test_split(
    X, Y_enc,
    test_size=0.2,
    random_state=42, # Zapewnia powtarzalność podziału
    stratify=Y_enc      # Ważne: dba o to, by w obu zbiorach był podobny % każdej klasy
)

print(f"Rozmiar X_train: {X_train.shape}")
print(f"Rozmiar X_val: {X_val.shape}")

# --- KROK 2: Konwersja na Tensory PyTorch ---

# BARDZO WAŻNE: Dodanie wymiaru "kanału" dla CNN
# Twój model CNN (Conv2D) oczekuje wejścia w formacie:
# (batch_size, kanały, wysokość, szerokość)
# Twoje X ma (ilość_clipów, wysokość, szerokość)
# Musimy dodać wymiar "1" dla kanału (jak obraz "czarno-biały")
# Używamy .unsqueeze(1)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

# Labele muszą być typu LongTensor (int64) dla funkcji straty CrossEntropyLoss
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

print(f"Nowy kształt tensora X_train: {X_train_tensor.shape}") # Powinno być (ilość, 1, 128, 401)

# --- KROK 3: Stworzenie `TensorDataset` ---
# Ten obiekt łączy Twoje spektrogramy (X) z ich etykietami (Y)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# --- KROK 4: Stworzenie `DataLoader` ---
# To jest "kelner", którego szukaliśmy.

BATCH_SIZE = 32 # Możesz eksperymentować z tą wartością

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,  # Kluczowe: MIESZAMY dane treningowe
    num_workers=2  # Opcjonalne: przyspiesza ładowanie danych
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, # Nie ma potrzeby mieszać danych walidacyjnych
    num_workers=2
)

# 2. Inicjalizacja modelu
# Pamiętaj, że 'in_channels=1' (dla spektrogramu "czarno-białego")
# i 'out_channels=6' (dla 6 gatunków muzycznych)
model = CNN(in_channels=1, out_channels=6)

# 3. Definicja funkcji straty (Kluczowe!)
# CrossEntropyLoss łączy LogSoftmax i NLLLoss.
# Dlatego model NIE powinien zwracać F.softmax()!
criterion = nn.CrossEntropyLoss()

# 4. Definicja optymalizatora
# Adam jest zazwyczaj dobrym i bezpiecznym wyborem na początek
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Ustawienie liczby epok
NUM_EPOCHS = 100

# 6. Uruchomienie pętli treningowej
trained_model = train_model(model,
                            train_loader,
                            val_loader,
                            criterion,
                            optimizer,
                            NUM_EPOCHS,
                            device)
torch.save(trained_model, "model_2.pt")

(2004, 128, 431)
(2004,)
['Bossa Nova' 'Funk' 'Jazz' 'Rock' 'Singer-Songwriter']
Rozmiar X_train: (1603, 128, 431)
Rozmiar X_val: (401, 128, 431)
Nowy kształt tensora X_train: torch.Size([1603, 1, 128, 431])
Rozpoczynanie treningu na cpu...
Epoch [1/100] | Train Loss: 1.8139 | Val Loss: 1.7032 | Val Accuracy: 26.93%
Epoch [2/100] | Train Loss: 1.7222 | Val Loss: 1.6793 | Val Accuracy: 29.43%
Epoch [3/100] | Train Loss: 1.6929 | Val Loss: 1.6021 | Val Accuracy: 30.42%
Epoch [4/100] | Train Loss: 1.6545 | Val Loss: 1.6015 | Val Accuracy: 26.93%
Epoch [5/100] | Train Loss: 1.6308 | Val Loss: 1.5495 | Val Accuracy: 28.43%
Epoch [6/100] | Train Loss: 1.5939 | Val Loss: 1.4832 | Val Accuracy: 30.67%
Epoch [7/100] | Train Loss: 1.5400 | Val Loss: 1.4155 | Val Accuracy: 33.92%
Epoch [8/100] | Train Loss: 1.4839 | Val Loss: 1.3349 | Val Accuracy: 35.41%
Epoch [9/100] | Train Loss: 1.4464 | Val Loss: 1.3450 | Val Accuracy: 52.62%
Epoch [10/100] | Train Loss: 1.4123 | Val Loss: 1.2303 | Val Accur

In [117]:
import random

model = torch.load("model_2.pt", weights_only=False)
# random_index = random.randint(0, len(X_train) - 1)
# random_sample = torch.tensor(np.array([X[random_index]]), dtype=torch.float32).unsqueeze(1)
# random_label = Y[random_index]

sample_array = np.load('original/bossa_nova_0.npy')
sample_array = sample_array.reshape(1, -1)[0]

sample_mel = get_mel_spectrogram(sample_array, sr=41000)
X_sample = np.array([sample_mel])

sample = torch.tensor(np.array([sample_mel]), dtype=torch.float32).unsqueeze(1)

prediction = model(sample)
_, pred_label = torch.max(prediction.data, 1)
print(label_encoder.classes_)
print(f"pred_label: {label_encoder.inverse_transform(pred_label)[0]}, random_label: Bossa Nova")

['Bossa Nova' 'Funk' 'Jazz' 'Rock' 'Singer-Songwriter']
pred_label: Bossa Nova, random_label: Bossa Nova
